In [1]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers

In [2]:
x = tf.random.normal((100000, 4))
y = tf.random.uniform((100000, 1), minval=0, maxval=2, dtype=tf.int32)

x = np.array(x).astype('float32') / 255.0
y = np.array(y)

print(x.shape)
print() 
print(y)
print() 
print(x)

(100000, 4)

[[0]
 [1]
 [1]
 ...
 [0]
 [0]
 [0]]

[[-0.00972259  0.00426545  0.00067111 -0.00370911]
 [-0.00480255 -0.002328    0.00011097  0.00404397]
 [ 0.00366527 -0.00148247 -0.00291658 -0.0033053 ]
 ...
 [-0.0057209   0.0016148  -0.00234543  0.00291655]
 [-0.00307552 -0.00249498  0.00185935 -0.00433112]
 [ 0.00319716  0.00269805  0.00232435 -0.0070193 ]]


I0000 00:00:1790273221.166312    3915 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790273221.171746    3915 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
from sklearn.model_selection import train_test_split as tts

x_train, x_test, y_train, y_test = tts(x, y, test_size=0.2, random_state=42)

print(x_train.shape)
print(x_test.shape)

(80000, 4)
(20000, 4)


In [4]:
import optuna
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping

In [5]:
def my_model(trial):
    lr = trial.suggest_float('lr', 1e-8, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1,15)


    model = keras.Sequential([
        keras.Input(shape=(4,))
    ])
    

    for i in range(n_layers):
        neurons = trial.suggest_int(f'neurons_{i}', 1,200, step=16)
        activation_fn = trial.suggest_categorical(f'activation_fn_{i}', ['relu', 'leaky_relu', 'elu', 'tanh'])
        initializer = trial.suggest_categorical(f'initializer_{i}', ['he_normal', 'glorot_normal'])
        dropouts = trial.suggest_float(f'dropouts_{i}', 0.0, 0.4, step=0.1)

        regularizer = trial.suggest_categorical(f'regularizer_{i}', ['l1','l2'])
        reg_rate = trial.suggest_float(f'reg_rate_{i}', 1e-8, 1e-2, log=True)
        if regularizer == 'l1':
            chosen_regularizer = regularizers.l1(reg_rate)
        else:
            chosen_regularizer = regularizers.l2(reg_rate)


        model.add(layers.Dense(neurons, kernel_initializer=initializer, kernel_regularizer=chosen_regularizer, activation=activation_fn))

        if dropouts > 0.0:
            model.add(layers.Dropout(dropouts))

    model.add(layers.Dense(1, activation='sigmoid'))

    
    model.compile(
        loss = keras.losses.BinaryCrossentropy(),
        optimizer = keras.optimizers.RMSprop(lr),
        metrics = ['accuracy']
    )

    early_stop = EarlyStopping(
        patience = 3,
        verbose = 1,
        restore_best_weights = True,
        monitor = 'val_loss'
    )

    history = model.fit(
        x_train, y_train,
        callbacks = [early_stop],
        epochs = 10,
        verbose = 1,
        batch_size = 128,
        validation_split = 0.2
    )

    trial.set_user_attr('trn_acc', history.history['accuracy'])
    trial.set_user_attr('trn_loss', history.history['loss'])
    trial.set_user_attr('val_acc', history.history['val_accuracy'])
    trial.set_user_attr('val_loss', history.history['val_loss'])

    model.save(f'model_trial_{trial.number}.keras')

    return max(history.history['val_accuracy'])



study = optuna.create_study(direction='maximize')
study.optimize(my_model, n_trials=5)

[I 2026-09-24 18:07:01,426] A new study created in memory with name: no-name-3bde2b04-95f7-4330-9bb7-8431f64849ac
/tmp/ipykernel_3915/1285522712.py:12: UserWarning: The distribution is specified by [1, 200] and step=16, but the range is not divisible by `step`. It will be replaced with [1, 193].
  neurons = trial.suggest_int(f'neurons_{i}', 1,200, step=16)


Epoch 1/10
 49/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5104 - loss: 12.3971

I0000 00:00:1790273233.668684    3978 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.4972 - loss: 12.1733 - val_accuracy: 0.4946 - val_loss: 11.9418
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4984 - loss: 11.7153 - val_accuracy: 0.4946 - val_loss: 11.4897
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4973 - loss: 11.2687 - val_accuracy: 0.4946 - val_loss: 11.0487
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5000 - loss: 10.8336 - val_accuracy: 0.4946 - val_loss: 10.6197
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4987 - loss: 10.4106 - val_accuracy: 0.5054 - val_loss: 10.2025
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4985 - loss: 9.9987 - val_accuracy: 0.4946 - val_loss: 9.7961
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4986 - loss: 9.5982 - val_accuracy: 0.4946 - val_loss: 9.4015
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4985 - loss: 9.2092 - val_accuracy: 0.

[I 2026-09-24 18:07:34,964] Trial 0 finished with value: 0.5053750276565552 and parameters: {'lr': 5.459551773305925e-06, 'n_layers': 14, 'neurons_0': 161, 'activation_fn_0': 'elu', 'initializer_0': 'glorot_normal', 'dropouts_0': 0.2, 'regularizer_0': 'l2', 'reg_rate_0': 6.83259755935319e-05, 'neurons_1': 161, 'activation_fn_1': 'relu', 'initializer_1': 'glorot_normal', 'dropouts_1': 0.0, 'regularizer_1': 'l1', 'reg_rate_1': 0.006150478857788758, 'neurons_2': 1, 'activation_fn_2': 'leaky_relu', 'initializer_2': 'glorot_normal', 'dropouts_2': 0.2, 'regularizer_2': 'l1', 'reg_rate_2': 7.293789858752205e-08, 'neurons_3': 33, 'activation_fn_3': 'leaky_relu', 'initializer_3': 'glorot_normal', 'dropouts_3': 0.4, 'regularizer_3': 'l1', 'reg_rate_3': 0.0011968615221588004, 'neurons_4': 193, 'activation_fn_4': 'leaky_relu', 'initializer_4': 'he_normal', 'dropouts_4': 0.30000000000000004, 'regularizer_4': 'l1', 'reg_rate_4': 1.0104135969501282e-06, 'neurons_5': 17, 'activation_fn_5': 'relu', 'in

...

[I 2026-09-24 18:08:27,230] Trial 2 finished with value: 0.4950000047683716 and parameters: {'lr': 6.395037851321486e-08, 'n_layers': 14, 'neurons_0': 113, 'activation_fn_0': 'elu', 'initializer_0': 'glorot_normal', 'dropouts_0': 0.30000000000000004, 'regularizer_0': 'l2', 'reg_rate_0': 0.0012133062333078437, 'neurons_1': 49, 'activation_fn_1': 'relu', 'initializer_1': 'he_normal', 'dropouts_1': 0.30000000000000004, 'regularizer_1': 'l1', 'reg_rate_1': 0.005199571526888769, 'neurons_2': 177, 'activation_fn_2': 'tanh', 'initializer_2': 'he_normal', 'dropouts_2': 0.4, 'regularizer_2': 'l1', 'reg_rate_2': 0.0023569446832305005, 'neurons_3': 177, 'activation_fn_3': 'leaky_relu', 'initializer_3': 'he_normal', 'dropouts_3': 0.0, 'regularizer_3': 'l2', 'reg_rate_3': 1.4096485724039263e-06, 'neurons_4': 97, 'activation_fn_4': 'elu', 'initializer_4': 'he_normal', 'dropouts_4': 0.30000000000000004, 'regularizer_4': 'l2', 'reg_rate_4': 0.0007850434560690335, 'neurons_5': 33, 'activation_fn_5': 't

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.4952 - loss: 1.8085 - val_accuracy: 0.4946 - val_loss: 1.8083
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5012 - loss: 1.8082 - val_accuracy: 0.4946 - val_loss: 1.8080
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5005 - loss: 1.8079 - val_accuracy: 0.4946 - val_loss: 1.8077
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5023 - loss: 1.8076 - val_accuracy: 0.4946 - val_loss: 1.8074
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4990 - loss: 1.8073 - val_accuracy: 0.4946 - val_loss: 1.8071
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5020 - loss: 1.8069 - val_accuracy: 0.4946 - val_loss: 1.8068
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4997 - loss: 1.8066 - val_accuracy: 0.4946 - val_loss: 1.8065
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5017 - loss: 1.8063 - val_accuracy: 0

[I 2026-09-24 18:09:01,118] Trial 3 finished with value: 0.4946250021457672 and parameters: {'lr': 8.371199802792871e-08, 'n_layers': 14, 'neurons_0': 17, 'activation_fn_0': 'leaky_relu', 'initializer_0': 'glorot_normal', 'dropouts_0': 0.2, 'regularizer_0': 'l2', 'reg_rate_0': 3.052653398966091e-05, 'neurons_1': 193, 'activation_fn_1': 'elu', 'initializer_1': 'he_normal', 'dropouts_1': 0.2, 'regularizer_1': 'l2', 'reg_rate_1': 0.0004105560089327929, 'neurons_2': 145, 'activation_fn_2': 'elu', 'initializer_2': 'he_normal', 'dropouts_2': 0.0, 'regularizer_2': 'l2', 'reg_rate_2': 1.3388362949229456e-07, 'neurons_3': 33, 'activation_fn_3': 'tanh', 'initializer_3': 'he_normal', 'dropouts_3': 0.30000000000000004, 'regularizer_3': 'l2', 'reg_rate_3': 0.004694153587641047, 'neurons_4': 129, 'activation_fn_4': 'relu', 'initializer_4': 'he_normal', 'dropouts_4': 0.4, 'regularizer_4': 'l1', 'reg_rate_4': 0.000454360044039676, 'neurons_5': 81, 'activation_fn_5': 'relu', 'initializer_5': 'he_normal

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.4995 - loss: 1.9550 - val_accuracy: 0.4954 - val_loss: 1.9519
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5001 - loss: 1.9488 - val_accuracy: 0.4959 - val_loss: 1.9458
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4999 - loss: 1.9427 - val_accuracy: 0.4952 - val_loss: 1.9396
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4994 - loss: 1.9366 - val_accuracy: 0.4958 - val_loss: 1.9335
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4968 - loss: 1.9305 - val_accuracy: 0.4952 - val_loss: 1.9275
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4992 - loss: 1.9244 - val_accuracy: 0.4957 - val_loss: 1.9214
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4999 - loss: 1.9183 - val_accuracy: 0.4966 - val_loss: 1.9153
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4980 - loss: 1.9123 - val_accuracy: 0.

[I 2026-09-24 18:09:21,742] Trial 4 finished with value: 0.49662500619888306 and parameters: {'lr': 1.2502500318466846e-06, 'n_layers': 5, 'neurons_0': 33, 'activation_fn_0': 'leaky_relu', 'initializer_0': 'he_normal', 'dropouts_0': 0.2, 'regularizer_0': 'l2', 'reg_rate_0': 7.148582478848451e-08, 'neurons_1': 113, 'activation_fn_1': 'leaky_relu', 'initializer_1': 'he_normal', 'dropouts_1': 0.0, 'regularizer_1': 'l1', 'reg_rate_1': 4.499764537037303e-07, 'neurons_2': 49, 'activation_fn_2': 'relu', 'initializer_2': 'he_normal', 'dropouts_2': 0.1, 'regularizer_2': 'l2', 'reg_rate_2': 7.958408497554977e-07, 'neurons_3': 33, 'activation_fn_3': 'tanh', 'initializer_3': 'glorot_normal', 'dropouts_3': 0.1, 'regularizer_3': 'l1', 'reg_rate_3': 0.006106766997906045, 'neurons_4': 161, 'activation_fn_4': 'leaky_relu', 'initializer_4': 'he_normal', 'dropouts_4': 0.2, 'regularizer_4': 'l2', 'reg_rate_4': 3.517758433225529e-06}. Best is trial 0 with value: 0.5053750276565552.


In [8]:
study.best_params

{'lr': 5.459551773305925e-06,
 'n_layers': 14,
 'neurons_0': 161,
 'activation_fn_0': 'elu',
 'initializer_0': 'glorot_normal',
 'dropouts_0': 0.2,
 'regularizer_0': 'l2',
 'reg_rate_0': 6.83259755935319e-05,
 'neurons_1': 161,
 'activation_fn_1': 'relu',
 'initializer_1': 'glorot_normal',
 'dropouts_1': 0.0,
 'regularizer_1': 'l1',
 'reg_rate_1': 0.006150478857788758,
 'neurons_2': 1,
 'activation_fn_2': 'leaky_relu',
 'initializer_2': 'glorot_normal',
 'dropouts_2': 0.2,
 'regularizer_2': 'l1',
 'reg_rate_2': 7.293789858752205e-08,
 'neurons_3': 33,
 'activation_fn_3': 'leaky_relu',
 'initializer_3': 'glorot_normal',
 'dropouts_3': 0.4,
 'regularizer_3': 'l1',
 'reg_rate_3': 0.0011968615221588004,
 'neurons_4': 193,
 'activation_fn_4': 'leaky_relu',
 'initializer_4': 'he_normal',
 'dropouts_4': 0.30000000000000004,
 'regularizer_4': 'l1',
 'reg_rate_4': 1.0104135969501282e-06,
 'neurons_5': 17,
 'activation_fn_5': 'relu',
 'initializer_5': 'he_normal',
 'dropouts_5': 0.1,
 'regulariz

In [16]:
best_model_num = study.best_trial.number
model = keras.models.load_model(f"model_trial_{best_model_num}.keras")

pred = model.predict(x_test)
pred

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


array([[0.5000679 ],
       [0.500068  ],
       [0.50006795],
       ...,
       [0.50006795],
       [0.50006795],
       [0.50006795]], dtype=float32)

In [17]:
# since as final output layers is set to sigmoid, hence it will only produce float numbers
# hence converting it into binary integer by applying a threshold

pred = (pred >= 0.5).astype(int)
print(pred)

[[1]
 [1]
 [1]
 ...
 [1]
 [1]
 [1]]


In [19]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_classes = np.argmax(pred, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred_classes))
print(classification_report(y_test, y_pred_classes))

Accuracy: 0.4994
              precision    recall  f1-score   support

           0       0.50      1.00      0.67      9988
           1       0.00      0.00      0.00     10012

    accuracy                           0.50     20000
   macro avg       0.25      0.50      0.33     20000
weighted avg       0.25      0.50      0.33     20000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
